In [ ]:
import pandas as pd

file_path = "../data/arXiv_scientific_dataset.csv" 

# Load the dataset
df = pd.read_csv(file_path)

# Quick look at the data structure
print("--- First 3 rows ---")
print(df.head(3)) 
# 2. Check column names, row count, and data types


print("\n--- Dataset Info ---")
df.info()

# 3. Check for missing values
print("\n--- Missing Values ---")
print(df.isnull().sum())

--- First 3 rows ---
             id                                              title  \
0  cs-9308101v1                               Dynamic Backtracking   
1  cs-9308102v1  A Market-Oriented Programming Environment and ...   
2  cs-9309101v1            An Empirical Analysis of Search in GSAT   

                  category category_code published_date updated_date  \
0  Artificial Intelligence         cs.AI         8/1/93       8/1/93   
1  Artificial Intelligence         cs.AI         8/1/93       8/1/93   
2  Artificial Intelligence         cs.AI         9/1/93       9/1/93   

                      authors      first_author  \
0          ['M. L. Ginsberg']  'M. L. Ginsberg'   
1           ['M. P. Wellman']   'M. P. Wellman'   
2  ['I. P. Gent', 'T. Walsh']      'I. P. Gent'   

                                             summary  summary_word_count  
0  Because of their occasional need to return to ...                  79  
1  Market price systems constitute a well-underst...  

In [ ]:
import os
import faiss
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

# 1. Create sample dataframe if not already in memory
sample_df = df.head(1000).copy()

# 2. Ensure the output directory exists
os.makedirs("../vector_store", exist_ok=True)

# 3. Select compute device for the GPU



device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# 4. Load the recommended embedding model onto the GPU
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)

# 5. Generate embeddings for the sample dataset using the summary field
print("Generating embeddings for sample dataset...")
summaries = sample_df["summary"].tolist()
embeddings = model.encode(
    summaries, 
    batch_size=256, 
    show_progress_bar=True, 
    convert_to_numpy=True
).astype("float32")

# 6. Build the FAISS vector database


print("Building FAISS index...")
dimension = embeddings.shape[1] 
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)
print(f"Successfully indexed {index.ntotal} vectors in FAISS!")

# 7. Save FAISS index and metadata
faiss.write_index(index, "../vector_store/arxiv_sample.index")
sample_df.to_pickle("../vector_store/arxiv_sample_metadata.pkl")
print("Saved sample FAISS index and metadata to '../vector_store/'.")

c:\Coursework\rag-research-assistant\venv\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


Using device: cuda


c:\Coursework\rag-research-assistant\venv\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Generating embeddings for sample dataset...


Batches: 100%|██████████| 4/4 [00:13<00:00,  3.28s/it]

Building FAISS index...
Successfully indexed 1000 vectors in FAISS!
Saved sample FAISS index and metadata to '../vector_store/'.


In [ ]:
import faiss
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder

# 1. Load FAISS index and metadata
index = faiss.read_index("../vector_store/arxiv_sample.index")
metadata_df = pd.read_pickle("../vector_store/arxiv_sample_metadata.pkl")

# 2. Set device
device = "cuda" if torch.cuda.is_available() else "cpu"

# 3. Load Bi Encoder (for initial retrieval) and Cross Encoder (for reranking)
bi_encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=device)




def search_and_rerank(query, top_k=20, final_top_n=3):
    # Step A: Dense Vector Search (FAISS)
    query_vector = bi_encoder.encode([query]).astype("float32")
    distances, indices = index.search(query_vector, top_k)
    
    # Extract retrieved candidate papers
    candidate_indices = indices[0]
    candidates = metadata_df.iloc[candidate_indices].copy()
    
    # Step B: Cross Encoder Reranking
    pairs = [[query, doc] for doc in candidates['summary']]
    rerank_scores = reranker.predict(pairs)
    
    # Add scores and sort by highest relevance
    candidates['rerank_score'] = rerank_scores
    reranked_results = candidates.sort_values(by='rerank_score', ascending=False).head(final_top_n)
    
    return reranked_results

# 4. Test Query
test_query = "machine learning models for search optimization"
results = search_and_rerank(test_query)

# Display top results


print(f"Top results for: '{test_query}'\n")
for idx, row in results.iterrows():
    print(f"Title: {row['title']}")
    print(f"Category: {row['category']} | Authors: {row['authors']}")
    print(f"Rerank Score: {row['rerank_score']:.4f}")
    print(f"Summary: {row['summary'][:150]}...\n" + "-"*50)

c:\Coursework\rag-research-assistant\venv\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange
c:\Coursework\rag-research-assistant\venv\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Top results for: 'machine learning models for search optimization'

Title: Genetic Programming for Kernel-based Learning with Co-evolving Subsets
  Selection
Category: Artificial Intelligence | Authors: ['Christian Gagné', 'Marc Schoenauer', 'Michèle Sebag', 'Marco Tomassini']
Rerank Score: 4.9032
Summary: Support Vector Machines (SVMs) are well-established Machine Learning (ML)
algorithms. They rely on the fact that i) linear learning can be formalized ...
--------------------------------------------------
Title: OPUS: An Efficient Admissible Algorithm for Unordered Search
Category: Artificial Intelligence | Authors: ['G. I. Webb']
Rerank Score: 3.5645
Summary: OPUS is a branch and bound search algorithm that enables efficient admissible
search through spaces for which the order of search operator application...
--------------------------------------------------
Title: Adaptive Parallel Iterative Deepening Search
Category: Artificial Intelligence | Authors: ['D. J. Cook', 'R. C. Varne

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# Load environment variables from .env
load_dotenv("../.env")

# Initialize OpenAI client with OpenRouter base URL
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

# Using the active free model tested successfully
LLM_MODEL = "nvidia/nemotron-3.5-lightning:free"

def format_context(results_df):
    """Formats the top reranked search results into structured prompt context."""
    context_blocks = []
    for idx, (_, row) in enumerate(results_df.iterrows(), 1):
        block = (
            f"Paper [{idx}]:\n"
            f"Title: {row['title']}\n"
            f"Authors: {row['authors']}\n"
            f"Category: {row['category']}\n"
            f"Summary: {row['summary']}\n"
        )
        context_blocks.append(block)
    return "\n".join(context_blocks)

def run_rag_pipeline(query, search_results):
    context = format_context(search_results)
    
    # -------------------------------------------------------------
    # PASS 1: Initial Answer Generation
    # -------------------------------------------------------------
    pass1_prompt = f"""You are an expert academic assistant. Answer the user's question using ONLY the provided research paper contexts.
Cite sources using paper numbers like [Paper 1], [Paper 2]. If the contexts do not contain enough information, state that clearly.

Context:
{context}

Question: {query}

Answer:"""

    print("--- Running Pass 1: Initial Generation ---")
    pass1_response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": pass1_prompt}],
        temperature=0.3
    )
    initial_answer = pass1_response.choices[0].message.content
    print("Pass 1 Output Generated.\n")

    # -------------------------------------------------------------
    # PASS 2: Grounding and Citation Refinement Pass
    # -------------------------------------------------------------
    pass2_prompt = f"""You are a strict editorial reviewer. Review the generated draft answer against the provided research contexts.

Task:
1. Ensure all claims are strictly grounded in the provided paper summaries.
2. Ensure citations are accurate and explicitly reference [Paper X] with title and author attribution where relevant.
3. Remove any unverified claims or hallucinations.
4. Format the final output cleanly in Markdown with a clear structure.

Contexts:
{context}



Draft Answer:
{initial_answer}

Refined Final Answer:"""

    print("--- Running Pass 2: Citation & Grounding Refinement ---")
    pass2_response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": pass2_prompt}],
        temperature=0.1
    )
    refined_answer = pass2_response.choices[0].message.content
    return refined_answer

# Execute test query against top reranked results
print("Executing two-pass pipeline...")
final_output = run_rag_pipeline(test_query, results)
print("\n=== FINAL REFINED RESPONSE ===")
print(final_output)

Executing two-pass pipeline...
--- Running Pass 1: Initial Generation ---
Pass 1 Output Generated.

--- Running Pass 2: Citation & Grounding Refinement ---

=== FINAL REFINED RESPONSE ===
**Review of "Machine Learning Models for Search Optimization" Against Provided Contexts**

Based on the supplied paper summaries, the topic of "machine learning models for search optimization" is not addressed as a broad survey but rather through specific, distinct case studies. Below is a clean, structured review ensuring all claims are strictly grounded, citations are accurate with title and author attribution, and unverified claims are removed.

### 1. Adaptive Parallel Search Strategy Selection  
**[Paper 3: Adaptive Parallel Iterative Deepening Search by D. J. Cook and R. C. Varnell]** describes the Eureka system, which employs machine learning techniques to automatically select optimal parallel search strategies for a given problem space. The system uses features describing the search space and 

In [5]:
import os
import faiss
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

# 1. Define file paths (adjusted for running inside the /notebooks directory)
data_path = "../data/arXiv_scientific_dataset.csv"
output_dir = "../vector_store"
index_path = os.path.join(output_dir, "arxiv_full.index")
metadata_path = os.path.join(output_dir, "arxiv_full_metadata.pkl")

# Ensure target directory exists
os.makedirs(output_dir, exist_ok=True)

# 2. Load full dataset
print("Loading full dataset...")
df = pd.read_csv(data_path)
print(f"Total rows loaded: {len(df):,}")

# 3. Detect compute device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using compute device: {device}")

# 4. Load model to GPU
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)

# 5. Generate embeddings for all paper summaries
print("Generating embeddings for all summaries (this will take a few minutes on GPU)...")
summaries = df["summary"].tolist()

embeddings = model.encode(
    summaries, 
    batch_size=512,  # Optimized for GPU throughput
    show_progress_bar=True, 
    convert_to_numpy=True
).astype("float32")

# 6. Build production FAISS index
print("Building FAISS index...")
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)
print(f"Successfully indexed {index.ntotal:,} vectors.")

# 7. Persist index and metadata dataframe
faiss.write_index(index, index_path)
df.to_pickle(metadata_path)
print(f"Saved full index to '{index_path}' and metadata to '{metadata_path}'.")

Loading full dataset...


'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: 596dd71c-ac91-4296-98b4-5e86f477bf5b)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json
Retrying in 1s [Retry 1/5].


Total rows loaded: 287,421
Using compute device: cuda


c:\Coursework\rag-research-assistant\venv\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Generating embeddings for all summaries (this will take a few minutes on GPU)...


Batches: 100%|██████████| 562/562 [17:47<00:00,  1.90s/it]


Building FAISS index...
Successfully indexed 287,421 vectors.
Saved full index to '../vector_store\arxiv_full.index' and metadata to '../vector_store\arxiv_full_metadata.pkl'.
